In [ ]:
# =========================
# 1) Imports
# =========================
import os
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import Dict, List, Tuple

import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from tqdm import tqdm

# pip install torchmetrics
from torchmetrics.detection.mean_ap import MeanAveragePrecision

In [ ]:
# =========================
# 2) Config
# =========================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CLASS_TO_ID = {
    "car": 1,
    "van": 2,
    "bus": 3,
    "others": 4
}
ID_TO_CLASS = {v: k for k, v in CLASS_TO_ID.items()}

# Paths (edit these)
XML_PATH = "path/to/MVI_20011.xml"
IMG_DIR  = "path/to/MVI_20011"           # contains img00001.jpg, ...
MODEL_PATH = "path/to/model.pt"          # your trained weights
BATCH_SIZE = 4
NUM_WORKERS = 2
CONF_THRESH = 0.30                       # prediction score threshold
IOU_MATCH_THRESH = 0.50                  # for Precision/Recall/F1 matching

In [ ]:
# =========================
# 3) Parse UA-DETRAC XML
# =========================
def parse_detrac_xml(xml_path: str) -> Dict[int, Dict[str, torch.Tensor]]:
    """
    Returns dict:
      frame_id -> {
        "boxes": FloatTensor [N,4] in x1,y1,x2,y2
        "labels": Int64Tensor [N]
      }
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()

    annotations = {}

    for frame in root.findall(".//frame"):
        fid = int(frame.get("num"))
        target_list = frame.find("target_list")

        boxes = []
        labels = []

        if target_list is not None:
            for target in target_list.findall("target"):
                attr = target.find("attribute")
                box = target.find("box")
                if attr is None or box is None:
                    continue

                vtype = attr.get("vehicle_type", "others").lower()
                if vtype not in CLASS_TO_ID:
                    vtype = "others"

                left = float(box.get("left", 0))
                top = float(box.get("top", 0))
                width = float(box.get("width", 0))
                height = float(box.get("height", 0))

                x1, y1 = left, top
                x2, y2 = left + width, top + height

                # filter invalid boxes
                if x2 <= x1 or y2 <= y1:
                    continue

                boxes.append([x1, y1, x2, y2])
                labels.append(CLASS_TO_ID[vtype])

        if len(boxes) > 0:
            annotations[fid] = {
                "boxes": torch.tensor(boxes, dtype=torch.float32),
                "labels": torch.tensor(labels, dtype=torch.int64)
            }

    return annotations

In [ ]:
# =========================
# 4) Dataset
# =========================
class DetracEvalDataset(Dataset):
    def __init__(self, img_dir: str, annotations: Dict[int, Dict[str, torch.Tensor]], transform=None):
        self.img_dir = Path(img_dir)
        self.annotations = annotations
        self.frame_ids = sorted(list(annotations.keys()))
        self.transform = transform if transform is not None else T.ToTensor()

    def __len__(self):
        return len(self.frame_ids)

    def __getitem__(self, idx):
        fid = self.frame_ids[idx]
        img_path = self.img_dir / f"img{fid:05d}.jpg"

        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            raise FileNotFoundError(f"Image not found: {img_path}")

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_tensor = self.transform(img_rgb)  # [C,H,W], float [0,1]

        target = {
            "boxes": self.annotations[fid]["boxes"].clone(),
            "labels": self.annotations[fid]["labels"].clone(),
            "image_id": torch.tensor([fid], dtype=torch.int64)
        }
        return img_tensor, target

def collate_fn(batch):
    images, targets = zip(*batch)
    return list(images), list(targets)

In [ ]:
# =========================
# 5) IoU + P/R/F1 helpers
# =========================
def box_iou_xyxy(boxes1: torch.Tensor, boxes2: torch.Tensor) -> torch.Tensor:
    """
    boxes1: [N,4], boxes2: [M,4]
    returns IoU matrix [N,M]
    """
    if boxes1.numel() == 0 or boxes2.numel() == 0:
        return torch.zeros((boxes1.shape[0], boxes2.shape[0]), dtype=torch.float32)

    area1 = (boxes1[:, 2] - boxes1[:, 0]).clamp(min=0) * (boxes1[:, 3] - boxes1[:, 1]).clamp(min=0)
    area2 = (boxes2[:, 2] - boxes2[:, 0]).clamp(min=0) * (boxes2[:, 3] - boxes2[:, 1]).clamp(min=0)

    lt = torch.max(boxes1[:, None, :2], boxes2[:, :2])   # [N,M,2]
    rb = torch.min(boxes1[:, None, 2:], boxes2[:, 2:])   # [N,M,2]
    wh = (rb - lt).clamp(min=0)                          # [N,M,2]
    inter = wh[..., 0] * wh[..., 1]                      # [N,M]

    union = area1[:, None] + area2 - inter
    return inter / union.clamp(min=1e-6)

@torch.no_grad()
def compute_precision_recall_f1(
    preds: List[Dict[str, torch.Tensor]],
    gts: List[Dict[str, torch.Tensor]],
    iou_thresh: float = 0.5,
    score_thresh: float = 0.3
):
    """
    Micro-averaged detection P/R/F1 over all classes.
    Matching is one-to-one per class using greedy score order.
    """
    TP, FP, FN = 0, 0, 0

    for pred, gt in zip(preds, gts):
        p_boxes = pred["boxes"].cpu()
        p_scores = pred["scores"].cpu()
        p_labels = pred["labels"].cpu()

        g_boxes = gt["boxes"].cpu()
        g_labels = gt["labels"].cpu()

        # confidence filter
        keep = p_scores >= score_thresh
        p_boxes = p_boxes[keep]
        p_scores = p_scores[keep]
        p_labels = p_labels[keep]

        # process per class
        classes = torch.unique(torch.cat([p_labels, g_labels], dim=0)) if (len(p_labels) or len(g_labels)) else torch.tensor([])
        for c in classes.tolist():
            pb = p_boxes[p_labels == c]
            ps = p_scores[p_labels == c]
            gb = g_boxes[g_labels == c]

            if len(pb) == 0 and len(gb) == 0:
                continue
            if len(pb) == 0:
                FN += len(gb)
                continue
            if len(gb) == 0:
                FP += len(pb)
                continue

            # sort predictions by confidence desc
            order = torch.argsort(ps, descending=True)
            pb = pb[order]

            ious = box_iou_xyxy(pb, gb)  # [P,G]
            matched_gt = torch.zeros(len(gb), dtype=torch.bool)

            for pi in range(len(pb)):
                iou_row = ious[pi]
                best_iou, best_gi = torch.max(iou_row, dim=0)
                if best_iou >= iou_thresh and not matched_gt[best_gi]:
                    TP += 1
                    matched_gt[best_gi] = True
                else:
                    FP += 1

            FN += (~matched_gt).sum().item()

    precision = TP / (TP + FP + 1e-9)
    recall = TP / (TP + FN + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)

    return {
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
# =========================
# 6) Load model
# =========================
def load_model(model_path: str, device: str):
    """
    Customize this for your model architecture.
    Option A: entire model saved with torch.save(model, path)
    Option B: state_dict saved with torch.save(model.state_dict(), path)
    """
    ckpt = torch.load(model_path, map_location=device)

    # ---- EDIT THIS BLOCK FOR YOUR TEAM'S MODEL ----
    # Example if entire model object was saved:
    if hasattr(ckpt, "eval"):
        model = ckpt
    else:
        # Example:
        # model = build_model(num_classes=5)  # background + 4 vehicle classes
        # model.load_state_dict(ckpt)
        raise ValueError("State dict detected. Please instantiate your model architecture before loading.")
    # -----------------------------------------------

    model.to(device)
    model.eval()
    return model

In [ ]:
# =========================
# 7) Main evaluation loop
# =========================
@torch.no_grad()
def evaluate_detector(model, dataloader, device="cuda", conf_thresh=0.3, iou_match_thresh=0.5):
    metric_map = MeanAveragePrecision(
        box_format="xyxy",
        iou_type="bbox",
        class_metrics=True
    )

    all_preds_for_f1 = []
    all_gts_for_f1 = []

    for images, targets in tqdm(dataloader, desc="Evaluating"):
        images = [img.to(device) for img in images]
        outputs = model(images)  # list[dict(boxes,scores,labels)]

        # move preds to cpu + score threshold for mAP input
        preds_batch = []
        gts_batch = []

        for out, tgt in zip(outputs, targets):
            boxes = out["boxes"].detach().cpu()
            scores = out["scores"].detach().cpu()
            labels = out["labels"].detach().cpu()

            keep = scores >= conf_thresh
            pred = {
                "boxes": boxes[keep],
                "scores": scores[keep],
                "labels": labels[keep]
            }

            gt = {
                "boxes": tgt["boxes"].detach().cpu(),
                "labels": tgt["labels"].detach().cpu()
            }

            preds_batch.append(pred)
            gts_batch.append(gt)

        metric_map.update(preds_batch, gts_batch)
        all_preds_for_f1.extend(preds_batch)
        all_gts_for_f1.extend(gts_batch)

    map_result = metric_map.compute()
    prf1_result = compute_precision_recall_f1(
        all_preds_for_f1,
        all_gts_for_f1,
        iou_thresh=iou_match_thresh,
        score_thresh=conf_thresh
    )

    return map_result, prf1_result

In [ ]:
# =========================
# 8) Run
# =========================
annotations = parse_detrac_xml(XML_PATH)
dataset = DetracEvalDataset(IMG_DIR, annotations, transform=T.ToTensor())
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn
)

model = load_model(MODEL_PATH, DEVICE)
map_result, prf1_result = evaluate_detector(
    model, loader, device=DEVICE,
    conf_thresh=CONF_THRESH,
    iou_match_thresh=IOU_MATCH_THRESH
)

print("===== Detection Metrics =====")
print(f"Precision: {prf1_result['precision']:.4f}")
print(f"Recall:    {prf1_result['recall']:.4f}")
print(f"F1-score:  {prf1_result['f1']:.4f}")
print(f"TP/FP/FN:  {prf1_result['TP']}/{prf1_result['FP']}/{prf1_result['FN']}")

print("\n===== mAP Metrics (torchmetrics) =====")
print(f"mAP@[0.50:0.95]: {map_result['map'].item():.4f}")
print(f"mAP@0.50:        {map_result['map_50'].item():.4f}")
print(f"mAP@0.75:        {map_result['map_75'].item():.4f}")
print(f"mAR@100:         {map_result['mar_100'].item():.4f}")

# optional class-wise AP
if "map_per_class" in map_result and map_result["map_per_class"] is not None:
    per_class = map_result["map_per_class"].cpu().numpy()
    classes = map_result["classes"].cpu().numpy()
    print("\nClass-wise AP:")
    for c, ap in zip(classes, per_class):
        cname = ID_TO_CLASS.get(int(c), f"class_{int(c)}")
        print(f"  {cname}: {ap:.4f}")